# Healthcare FHIR Integration : HL7 v2 vers FHIR

**Entrée** : `hl7/sample_message.hl7`, message d'admission ADT^A01 **fictif**  
**Sortie** : un dictionnaire FHIR `Patient` en mémoire (aucun fichier patient n'est écrit)  
**Stack** : Python (bibliothèque standard), module `hl7/hl7_to_fhir.py`

---

### Objectif

Beaucoup d'hôpitaux émettent encore du HL7 v2 (segments séparés par `|`), alors que les systèmes récents utilisent FHIR. Ce notebook montre la traduction du segment `PID` (identifiant, nom, date de naissance, genre) vers un `Patient` FHIR.

### Fonctionnement

1. **Lecture** du message et isolement du segment `PID`.
2. **Mapping déterministe**, sans LLM : une date ou un genre mal converti dans un dossier patient n'est pas acceptable. `PID-3` devient `Patient.identifier` (et non `Patient.id`), `PID-5` le nom, `PID-7` la date de naissance, `PID-8` le genre.
3. **Cas limites explicites** : dates partielles (`1992`, `199204`) conservées, date impossible (`20260231`) rejetée, code de genre inattendu converti en `unknown` avec un avertissement.

### Limite assumée

Seul le segment `PID` d'un ADT simulé est traité. Ce n'est pas un moteur d'intégration HL7 complet.

## 1. Chargement du message

On importe les fonctions du module `hl7_to_fhir`, puis on lit l’exemple versionné. Le contenu complet n’est volontairement pas affiché, conformément au principe de minimisation des données dans les logs.


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hl7.hl7_to_fhir import (
    convert_birth_date,
    convert_gender,
    find_pid_segment,
    parse_pid,
    read_hl7_message,
)

message_path = PROJECT_ROOT / "hl7" / "sample_message.hl7"
message = read_hl7_message(message_path)
print(f"Message chargé : {len(message.splitlines())} segments")


Message chargé : 2 segments


> **Contrôle.** Le fichier est lu depuis un chemin relatif au dépôt et l’affichage se limite au nombre de segments. Le notebook reste portable et ne révèle pas les valeurs du message dans ses logs.


## 2. Extraction du PID et conversion

Le segment PID porte l’identité administrative. Le parseur mappe `PID-3` vers `Patient.identifier`, le nom vers `Patient.name`, puis normalise la date et le genre.


In [2]:
pid_segment = find_pid_segment(message)
fhir_patient = parse_pid(pid_segment)

print(json.dumps(fhir_patient, indent=2, ensure_ascii=False))


{
  "resourceType": "Patient",
  "identifier": [
    {
      "value": "PAT12345"
    }
  ],
  "name": [
    {
      "family": "MARTIN",
      "given": [
        "Julie"
      ]
    }
  ],
  "gender": "female",
  "birthDate": "1992-04-03"
}


> **Résultat.** Une ressource `Patient` structurée est produite. L’identifiant métier HL7 reste dans `Patient.identifier` et n’est pas confondu avec `Patient.id`, qui représente l’identifiant logique attribué par un serveur FHIR.


## 3. Cas limites des règles déterministes

On vérifie les dates partielles autorisées par FHIR, une date impossible et les principaux codes de genre. Ces règles sont testables et ne nécessitent aucune interprétation générative.


In [3]:
date_cases = ["1992", "199204", "19920403", "20260231", ""]
gender_cases = ["F", "M", "O", "U", "X", ""]

print("Dates :", {value or "(vide)": convert_birth_date(value) for value in date_cases})
print("Genres :", {value or "(vide)": convert_gender(value) for value in gender_cases})


Date de naissance HL7 impossible : valeur ignorée


Code de sexe HL7 inattendu 'X' : normalisé en 'unknown'


Dates : {'1992': '1992', '199204': '1992-04', '19920403': '1992-04-03', '20260231': None, '(vide)': None}
Genres : {'F': 'female', 'M': 'male', 'O': 'other', 'U': 'unknown', 'X': 'unknown', '(vide)': 'unknown'}


> **Observation.** Les précisions année, mois et jour sont conservées quand elles sont valides. Une date impossible devient `None`. Les codes connus sont mappés explicitement ; un code inattendu devient `unknown` avec avertissement, tandis qu’un champ vide représente simplement une absence d’information.

## Conclusion et décisions pour la suite

Le mapping HL7 → FHIR est transparent, reproductible et indépendant du LLM. L’assistant local n’intervient qu’après une erreur déjà détectée par Python, pour l’expliquer en langage naturel.

**Décisions :** garder la sévérité sous contrôle du code ; compléter ultérieurement la gestion des séparateurs, répétitions et échappements HL7 ; mapper l’autorité d’attribution de `PID-3` vers `identifier.system`.
